# DiffusionGemma-Jev (`djev`) on Google Colab Pro (`A100` / `L4`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taeold/djev-run/blob/main/colab.ipynb)

> **Colab Pro `A100` / `L4` Required by Default:** This notebook requests an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** (`gpuClass: premium`, `machine_shape: hm`) so all `17.53 GiB` of weights stay 100% in GPU VRAM (`~45 ms` on A100 HBM2e, `~65 ms` on L4). If Colab connects you to a default `T4`, click **`Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU or L4 GPU`** (or top-right dropdown arrow next to `T4 -> Change runtime type`).

Run **DiffusionGemma-Jev** (`nvidia/diffusiongemma-26B-A4B-it-NVFP4`, `26B` total parameters, `4B` active per token across `128` experts) directly inside Google Colab Pro on an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** GPU using standard `vLLM` (`POST /tokenize` + `POST /v1/chat/completions` with `extra_body.vllm_xargs`), and play the built-in 1-step diffusion games (`/tetris`, `/dino`, and `/snake`) live inside the notebook.

| Colab Runtime | GPU / Compute Capability | Usable VRAM | `nvidia/diffusiongemma-26B-A4B-it-NVFP4` (`17.53 GiB`) | Active vLLM Kernel Path |
| :--- | :--- | :--- | :--- | :--- |
| **Colab Pro (`A100`)** | NVIDIA A100 (`SM 8.0`) | `40.0 GiB` / `80.0 GiB` | **Recommended (`~45 ms/step`, 100% VRAM)** (`KV_CACHE_GB=8`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Pro (`L4`)** | NVIDIA L4 (`SM 8.9`) | `22.5 GiB` (`24 GB`) | **Supported (`~65 ms/step`, 100% VRAM)** (`KV_CACHE_GB=1.5`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Free (`T4`)** | NVIDIA T4 (`SM 7.5`) | `15.0 GiB` (`16 GB`) | **Blocked by default** (`ALLOW_SLOW_T4_OFFLOAD = False`; requires `5 GB` PCIe CPU offload at `~400-600 ms/step`) | Switch to `A100` or `L4` in `Runtime -> Change runtime type` |

In [ ]:
!pip uninstall -y torchaudio torchvision && pip install -U vllm --pre --extra-index-url https://wheels.vllm.ai/nightly

## Step 1: Start Server (`server.py`)

This cell downloads the latest `server.py` and demo HTMLs, and spins up `vllm serve` natively on Colab's GPU in the background.

In [ ]:
import os
import subprocess
import urllib.request
import json
import time

DJEV_BASE_URL = "http://127.0.0.1:8080"
if not os.path.exists("server.py"):
    for f in ["server.py", "snake.html", "dino.html", "tetris.html"]:
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/taeold/djev-run/main/{f}", f)

print("Starting vLLM server in background... (Wait ~3m for PyTorch kernels)")
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
proc = subprocess.Popen(
    "python3 server.py".split(),
    stdout=open("/tmp/vllm.log", "w"), stderr=subprocess.STDOUT
)
while True:
    try:
        if json.loads(urllib.request.urlopen(f"{DJEV_BASE_URL}/health", timeout=1).read().decode())["name"] == "vllm":
            print("\n[djev] Server is ready on Colab GPU!")
            break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)


## Step 2: 1-Step Diffusion Canvas Read via Standard `vLLM` (`POST /tokenize` + `POST /v1/chat/completions`)

Tokenize an output template (`POST /tokenize`), pin every scaffold and label token (`diffusion_pinned`), leave the answer slots unpinned (`department`, `urgency`, `refund_requested`), and read all three slot probability distributions simultaneously in **1 forward pass** (`diffusion_max_steps: 1`, `diffusion_read_only: True`).

In [ ]:
import json
import math
import time
import urllib.request

SCAFFOLD = [100, 45518, 107, 101]  # <thought>\n</thought>
template_str = "department: a\nurgency: 1\nrefund_requested: yes"

# 1. Tokenize template via POST /tokenize
tok_req = urllib.request.Request(
    f"{DJEV_BASE_URL}/tokenize",
    data=json.dumps({"prompt": template_str, "add_special_tokens": False}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer mock"},
)
base_ids = json.loads(urllib.request.urlopen(tok_req, timeout=10).read().decode())["tokens"]
full_template = SCAFFOLD + base_ids

unpinned = [i - 1 for i, t in enumerate(full_template) if i >= 4 and t == 107] + [len(full_template) - 1]
pinned = [i for i in range(len(full_template)) if i not in unpinned]
# Canvas padded to 128 elements securely
seed_canvas = [full_template[i] if i in pinned else (256000 + i * 131) for i in range(len(full_template))]
seed_canvas = seed_canvas + [256000] * (128 - len(seed_canvas))

# 2. Run 1-step read-only diffusion forward pass via POST /v1/chat/completions
chat_payload = {
    "model": "djev-dgemma",
    "messages": [
        {
            "role": "system",
            "content": (
                "Answer each question about the ticket state with its single label.\n"
                "department: a = billing, b = technical, c = sales\n"
                "urgency: 1 = low, 2 = minor, 3 = locked production access, 4 = complete outage\n"
                "refund_requested: yes or no"
            ),
        },
        {
            "role": "user",
            "content": json.dumps({
                "ticket_id": "TCK-9042",
                "customer_tier": "enterprise",
                "text": "I was double-charged $149.00 on invoice INV-2026-8841 and my production API key is locked. Please refund the duplicate charge.",
            }),
        },
    ],
    "max_tokens": len(full_template) + 1,
    "logprobs": True,
    "top_logprobs": 16,
    "vllm_xargs": {
        "diffusion_seed_canvas": seed_canvas,
        "diffusion_pinned": pinned,
        "diffusion_max_steps": 1,
        "diffusion_read_only": True,
    }
}

t0 = time.time()
req = urllib.request.Request(
    f"{DJEV_BASE_URL}/v1/chat/completions",
    data=json.dumps(chat_payload).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer mock"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=30).read().decode())
rtt_ms = round((time.time() - t0) * 1000, 1)
content_lp = resp["choices"][0]["logprobs"]["content"]

def slot_softmax(pos, label_map):
    top = {entry["token"].strip(): entry["logprob"] for entry in content_lp[pos]["top_logprobs"]}
    lps = [top.get(lbl, -20.0) for lbl in label_map]
    mx = max(lps)
    ex = [math.exp(x - mx) for x in lps]
    s = sum(ex)
    return {name: ex[i] / s for i, name in enumerate(label_map.values())}

dept_probs = slot_softmax(unpinned[0], {"a": "billing", "b": "technical", "c": "sales"})
urg_probs = slot_softmax(unpinned[1], {"1": "1", "2": "2", "3": "3", "4": "4"})
ref_probs = slot_softmax(unpinned[2], {"yes": "yes", "no": "no"})

best_dept = max(dept_probs, key=dept_probs.get)
exp_urg = sum(int(k) * v for k, v in urg_probs.items())
best_urg = max(urg_probs, key=urg_probs.get)

print(f"1-Step Diffusion Canvas Read Complete in {rtt_ms} ms")
print("-" * 78)
print(f"{'SLOT ID':<18} | {'TYPE':<8} | {'PREDICTION':<22} | {'CONFIDENCE / SCORE'}")
print("-" * 78)
print(f"{'department':<18} | {'choice':<8} | {best_dept:<22} | {dept_probs[best_dept]*100:5.1f}% (probs: {json.dumps({k: round(v, 3) for k, v in dept_probs.items()})})")
print(f"{'urgency':<18} | {'score':<8} | {'level ' + best_urg + '/4':<22} | {exp_urg:.2f} / 4.00 (conf: {urg_probs[best_urg]*100:5.1f}%)")
print(f"{'refund_requested':<18} | {'bool':<8} | {str(ref_probs['yes'] >= 0.5):<22} | P(yes) = {ref_probs['yes']*100:5.1f}%")
print("-" * 78)


## Step 3: Play `/tetris`, `/dino`, and `/snake` Live Inside Colab

Set `DEMO = "/tetris"`, `"/dino"`, or `"/snake"` and run the cell below to embed the live game UI served from port `8080` on your Colab GPU.

In [ ]:
# Choose a built-in game: "/tetris", "/dino", or "/snake"
DEMO = "/tetris"

try:
    from google.colab import output
    print(f"Embedding {DEMO} from Colab GPU server (port 8080)...")
    output.serve_kernel_port_as_iframe(8080, path=DEMO, height=660)
except ImportError:
    print("Available demo routes on local server:")
    for r in ("/tetris", "/dino", "/snake"):
        print(f"  -> {DJEV_BASE_URL}{r}")
